# Pilot preflight and detached execution control

Purpose: actual-pilot batching/checkpoint recovery and authorized execution. Inputs: pinned data, pilot configuration, launch manifest. Outputs: preflight certificate, executed pilot notebook and certified handoff. Definitions only on import.

Expected runtime: definitions seconds; preflight1–5minutes; full pilot hours. Checkpoints: checkpoints/<run_id>-<kind>/, latest two plus best. Exact resume: rerun the associated pilot_driver.ipynb with unchanged run ID, configuration and source. Invalid manifests raise; one driver and one model writer are enforced.

In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:

from pathlib import Path
import copy, datetime, gc, json, os, time, uuid
import nbformat
import torch
from fmm_jepa.training import atomic_json,definition_hashes,_assert_equal_tree,digest_file
from fmm_jepa.tiny_training import initialize_paired_models,run_model,validate_training_manifest,_rng,_restore_rng
from fmm_jepa.tiny_workflow import resolve_config,start_run,certify_clean_execution,identity_hash
from fmm_jepa.tiny_imagenet import prepare_dataset
from fmm_jepa.geometry import clear_geometry_cache
PILOT_CONTROL_SCHEMA='fmm-jepa-pilot-control/v1'

def _pilot_exact_tree(a,b):
    import numpy as np
    if isinstance(a,np.ndarray):assert isinstance(b,np.ndarray) and np.array_equal(a,b),'NumPy RNG arrays differ'
    elif torch.is_tensor(a):assert torch.is_tensor(b) and torch.equal(a,b),'Tensor states differ'
    elif isinstance(a,dict):
        assert a.keys()==b.keys()
        for key in a:_pilot_exact_tree(a[key],b[key])
    elif isinstance(a,(list,tuple)):
        assert type(a)==type(b) and len(a)==len(b)
        for x,y in zip(a,b):_pilot_exact_tree(x,y)
    else:assert a==b,f'States differ: {a!r} != {b!r}'


def pilot_preflight(root,artifact,updates=4,split_at=2):
    """Actual pilot architecture/batch/dtype/device, full5000 schedule; no production outcomes."""
    root=Path(root);artifact=Path(artifact);artifact.mkdir(parents=True,exist_ok=True)
    if (updates,split_at)!=(4,2):raise ValueError('Certificate is registered K4,J2')
    config=resolve_config(root,'pilot')
    if config['device']=='cpu':raise RuntimeError('Actual-pilot GPU preflight requires CUDA')
    assert config['training']['updates']==5000 and config['effective_batch_size']==128
    assert config['evaluation']['heldout_enabled'] is False
    bundle=prepare_dataset(root,config)
    models,counts,initialization=initialize_paired_models(config,artifact)
    protocol={'schema':PILOT_CONTROL_SCHEMA,'purpose':'actual-pilot batch/checkpoint preflight, not representation comparison',
              'source_hashes':definition_hashes(root),'configuration':config,'parameter_counts':counts,
              'split_sha256':bundle['split_manifest']['split_sha256'],'K':updates,'J':split_at,
              'preserve_final_schedule':5000,'global_rng_reset_before_paired_trajectories':True}
    atomic_json(artifact/'resolved_config.json',config);atomic_json(artifact/'protocol.json',protocol)
    original_rng=_rng();results={};preflight_id=artifact.name
    try:
        for kind in ('vit','fmm'):
            print(kind+': actual pilot batch8 x accumulation16; uninterrupted4 vs resumed2+2',flush=True)
            start_rng=copy.deepcopy(original_rng)
            reference_dir=root/'checkpoints'/(preflight_id+'-'+kind+'-reference')
            split_dir=root/'checkpoints'/(preflight_id+'-'+kind+'-split')
            _restore_rng(start_rng)
            reference=run_model(kind,config,bundle,artifact/(kind+'-reference'),reference_dir,protocol,model=copy.deepcopy(models[kind]),stop_after=updates)
            ref_manifest=validate_training_manifest(reference_dir)
            ref=torch.load(reference_dir/ref_manifest['latest'],map_location='cpu',weights_only=False)
            performance=copy.deepcopy(reference['performance']);del reference;clear_geometry_cache();gc.collect();torch.cuda.empty_cache()
            _restore_rng(start_rng)
            partial=run_model(kind,config,bundle,artifact/(kind+'-split'),split_dir,protocol,model=copy.deepcopy(models[kind]),stop_after=split_at)
            assert partial['resume']['final_step']==split_at
            del partial;clear_geometry_cache();gc.collect();torch.cuda.empty_cache()
            resumed=run_model(kind,config,bundle,artifact/(kind+'-split'),split_dir,protocol,model=copy.deepcopy(models[kind]),stop_after=updates)
            assert resumed['resume']['resumed_from_step']==split_at
            split_manifest=validate_training_manifest(split_dir)
            continued=torch.load(split_dir/split_manifest['latest'],map_location='cpu',weights_only=False)
            checked=['model','optimizer','scheduler','ema_position','scaler','rng','sampler','streams','config','protocol','identity']
            for field in checked:_pilot_exact_tree(ref[field],continued[field])
            _pilot_exact_tree(ref['state']['traces'],continued['state']['traces'])
            _pilot_exact_tree([r['loss'] for r in ref['state']['metrics']],[r['loss'] for r in continued['state']['metrics']])
            assert ref['state']['image_exposures']==continued['state']['image_exposures']==updates*128
            assert ref['sampler']['cursor']==continued['sampler']['cursor']==updates*128
            assert len(split_manifest['checkpoints'])>=2
            results[kind]={'status':'PASS','bitwise_equal':True,'successful_updates':updates,'split_at':split_at,
                           'exposures_per_trajectory':updates*128,'microbatch':8,'accumulation_steps':16,
                           'device':config['device'],'dtype':'float32','model':config['model'],'schedule_budget':5000,
                           'compared':checked+['sample/occurrence/augmentation/mask traces','losses'],
                           'reference_manifest':str(reference_dir/'manifest.json'),'resumed_manifest':str(split_dir/'manifest.json'),
                           'latest_two_valid':True,'performance':performance}
            atomic_json(artifact/'preflight_partial.json',results)
            print(kind+': exact recovery PASS; median step '+str(performance['complete_jepa_step_seconds']['median'])+' seconds',flush=True)
            del ref,continued,resumed;clear_geometry_cache();gc.collect();torch.cuda.empty_cache()
    finally:_restore_rng(original_rng)
    estimates={k:5000*v['performance']['complete_jepa_step_seconds']['median'] for k,v in results.items()}
    report={'schema':PILOT_CONTROL_SCHEMA,'status':'PASS','preflight_id':preflight_id,'models':results,
            'expected_training_seconds':estimates,'estimated_pair_training_hours':sum(estimates.values())/3600,
            'estimated_pair_with_overhead_hours':sum(estimates.values())*1.25/3600,
            'estimate_limit':'Two post-warmup updates per model; excludes later feature/probe work,25percent overhead is an allowance, not guarantee.',
            'requires_detached_tmux':True,'heldout_enabled':False,'source_hashes':protocol['source_hashes']}
    atomic_json(artifact/'preflight_certificate.json',report);return report


In [ ]:

def make_pilot_notebook(root,output_path,preflight_path):
    """Copy the tested main workflow; adjust explanatory markdown for the active pilot only."""
    root=Path(root);source=root/'notebooks/01_TinyImageNet_Debug.ipynb';n=nbformat.read(source,4)
    for cell in n.cells:
        if cell.cell_type=='code':cell.outputs=[];cell.execution_count=None
        else:
            cell.source=cell.source.split('### Executed smoke readout')[0].rstrip()
            cell.source=cell.source.replace('One paired seed and a short smoke cannot establish an architectural advantage.','One paired pilot seed cannot establish a robust architectural advantage; intervals exclude training-seed variability.')
            cell.source=cell.source.replace('# 01 · Tiny ImageNet: learning and pipeline verification','# 01 · Tiny ImageNet: registered paired pilot')
            if cell.source.startswith('## 13'):
                cell.source='## 13 · Execute the authorized paired pilot\n\nThe active pilot runs5,000 successful updates per model with microbatch8 and accumulation16,effectivebatch128. Each model consumes640,000 exposures from the same20,000-image training subset. Consumed IDs, occurrences, augmentations and masks are paired. Schedules and final-checkpoint selection are unchanged. Models run sequentially on one GPU.\n\nFixed raw diagnostics use one development image/class at0,2,500,5,000steps. Checkpoints save every100updates, epoch boundary and at most15minutes. A detached tmux session owns the run.'
            if cell.source.startswith('## 14'):
                cell.source='## 14 · Active pilot configuration\n\nThis execution explicitly selects tiny_imagenet_pilot.yaml:20,000training/2,000development examples,5,000updates/model,effectivebatch128,and30epochs for allfour frozen probes. The same implementation passed smoke and actual-pilot batch/resume preflight. Full-pool and held-out prediction remain disabled. Interrupted execution resumes validated manifests.'
            if cell.source.startswith('## 22'):
                cell.source='## 22 · Pilot handoff and decision\n\nThe handoff separates engineering readiness, trained-versus-initial learning evidence and readiness for Food-101. Recommendations consider completed pilot results and raw diagnostics; FMM need not beat ViT. No later-stage dataset or notebook is launched here. Any future Food-101 comparison starts fresh unless a separate transfer protocol is authorized.\n\nIndependent review checks raw scores, paired uncertainty, covariance concentration, branch activity and compute. Held-out evaluation remains NOT_RUN. External execution closes E14 only after all cells finish.'
    n.cells[0].source+='\n\n**Active execution is the authorized pilot.** Preflight: `'+str(preflight_path)+'`. Canonical smoke outputs are preserved. Exact resume uses the associated pilot_driver.ipynb with unchanged run ID/config/source. The estimated multi-hour training runs inside detached tmux.'
    for index,cell in enumerate(n.cells):cell.id=f'pilot-{index:03d}'
    nbformat.write(n,output_path);return n

def execute_pilot(root,run_id):
    """Detached driver: fresh main kernel, partial notebook snapshots, external certification."""
    import fcntl,traceback
    from nbclient import NotebookClient
    root=Path(root);artifact=root/'artifacts/notebook_01'/run_id
    launch=json.loads((artifact/'pilot_launch.json').read_text())
    preflight=json.loads(Path(launch['preflight_certificate']).read_text())
    if preflight['status']!='PASS':raise ValueError('Preflight not PASS')
    if preflight['source_hashes']!=launch['source_hashes']:raise ValueError('Preflight and production sources differ')
    if definition_hashes(root)!=launch['source_hashes']:raise ValueError('Source changed after pilot preparation')
    config=resolve_config(root,'pilot');start_run(root,config,run_id)
    path=artifact/'01_TinyImageNet_Pilot.ipynb'
    lock=(artifact/'.pilot-driver.lock').open('a+')
    try:fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    except BlockingIOError:lock.close();raise RuntimeError('Another pilot driver is active')
    previous={key:os.environ.get(key) for key in ('FMM_JEPA_ROOT','FMM_JEPA_01_PROFILE','FMM_JEPA_01_RESUME')}
    os.environ.update(FMM_JEPA_ROOT=str(root),FMM_JEPA_01_PROFILE='pilot',FMM_JEPA_01_RESUME=run_id)
    started=time.perf_counter();started_utc=datetime.datetime.now(datetime.timezone.utc).isoformat()
    notebook=None;log_path=artifact/'pilot_execution.log'
    def log(message):
        with log_path.open('a') as stream:stream.write(datetime.datetime.now(datetime.timezone.utc).isoformat()+' '+message+'\n');stream.flush()
        print(message,flush=True)
    def save_notebook():
        if notebook is None:return
        temp=path.with_suffix('.tmp.ipynb');nbformat.write(notebook,temp);os.replace(temp,path)
    def start_cell(cell,cell_index,**kwargs):
        if cell.cell_type=='code':
            status={'schema':PILOT_CONTROL_SCHEMA,'status':'RUNNING','run_id':run_id,'driver_pid':os.getpid(),'started_utc':started_utc,
                    'cell_index':cell_index,'cell_first_line':cell.source.splitlines()[0],'elapsed_seconds':time.perf_counter()-started,'tmux_session':launch['tmux_session']}
            atomic_json(artifact/'pilot_status.json',status);log('CELL '+str(cell_index)+' '+status['cell_first_line'])
    def end_cell(cell,cell_index,**kwargs):
        for output in cell.get('outputs',[]):
            if output.get('output_type')=='stream' and output.get('text','').strip():log(output['text'].strip())
        save_notebook()
    try:
        notebook=make_pilot_notebook(root,path,launch['preflight_certificate'])
        log('BEGIN '+run_id+' full registered pilot; session '+launch['tmux_session'])
        client=NotebookClient(notebook,timeout=None,kernel_name='ahr-go',resources={'metadata':{'path':str(root)}},on_cell_start=start_cell,on_cell_executed=end_cell)
        client.execute();save_notebook()
        certificate=certify_clean_execution(root,artifact,path,time.perf_counter()-started)
        handoff=json.loads((artifact/'handoff_01_to_02.json').read_text())
        if not handoff['engineering_ready']:raise RuntimeError('An engineering gate remains unclosed')
        atomic_json(artifact/'pilot_status.json',{'schema':PILOT_CONTROL_SCHEMA,'status':'COMPLETED','run_id':run_id,'started_utc':started_utc,
            'finished_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'elapsed_seconds':time.perf_counter()-started,
            'tmux_session':launch['tmux_session'],'clean_execution':certificate['status'],'learning_evidence':handoff['learning_evidence'],
            'independent_interpretation_review':'PENDING'})
        log('COMPLETED '+run_id+'; all engineering gates PASS; learning='+handoff['learning_evidence'])
        return {'run_id':run_id,'artifact_directory':str(artifact),'status':'COMPLETED','learning_evidence':handoff['learning_evidence']}
    except BaseException as error:
        save_notebook();atomic_json(artifact/'pilot_status.json',{'schema':PILOT_CONTROL_SCHEMA,'status':'FAILED','run_id':run_id,
            'error':repr(error),'traceback':traceback.format_exc(),'elapsed_seconds':time.perf_counter()-started,
            'resume':'Rerun driver with unchanged run_id/config/source; inspect checkpoint manifests first.'})
        log('FAILED '+repr(error));raise
    finally:
        for key,value in previous.items():
            if value is None:os.environ.pop(key,None)
            else:os.environ[key]=value
        fcntl.flock(lock,fcntl.LOCK_UN);lock.close()
